<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/faceSwap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face swap sur images fixes

Dans ce notebook, on réalise un **face swap** entre deux photos.

## Idée principale
On prend :
- une **image source** : le visage que l'on veut transférer ;
- une **image cible** : l'image dans laquelle on veut insérer ce visage.

Le modèle détecte les visages, puis essaie de transférer l'identité du visage source sur le visage cible.

## Ce qu'on veut comprendre
- ce qu'est un face swap ;
- ce qui est réellement modifié dans l'image ;
- quels artefacts apparaissent ;
- quelles limites techniques et éthiques cela pose.

## Règles du notebook

Ce notebook doit être utilisé uniquement avec :
- vos propres photos ;
- ou des photos pour lesquelles vous avez un consentement explicite.

## Important
Le résultat doit être présenté comme un **média synthétique**.
Nous ajouterons donc un marquage visible sur l'image finale.

In [ ]:
!nvidia-smi || true

import sys
import platform

print("Python :", sys.version)
print("Plateforme :", platform.platform())

## Installer les bibliothèques utiles

On installe ici :
- `insightface` pour la détection et l'alignement des visages ;
- `onnxruntime-gpu` pour l'inférence ;
- les bibliothèques d'image et d'affichage.

In [ ]:
!pip -q install insightface onnxruntime-gpu opencv-python pillow matplotlib huggingface_hub

## Télécharger le modèle de face swap

Nous utilisons ici le modèle `inswapper_128.onnx`.

### Remarque
Le téléchargement automatique peut parfois varier selon les environnements.
Si besoin, on pourrait aussi le téléverser manuellement.

In [ ]:
import os
from huggingface_hub import hf_hub_download

MODEL_REPO = "thebiglaskowski/inswapper_128.onnx"
MODEL_FILENAME = "inswapper_128.onnx"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILENAME)
print("Modèle téléchargé :", model_path)

## Téléverser deux images

Vous devez maintenant fournir :
- une **image source** : le visage à transférer ;
- une **image cible** : l'image sur laquelle appliquer le swap.

## Conseils
Pour une première démonstration :
- visage bien visible ;
- photo nette ;
- peu d'occlusions ;
- cadrage assez simple.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Fichiers reçus :", list(uploaded.keys()))

In [ ]:
image_extensions = (".png", ".jpg", ".jpeg", ".webp")
images = [f for f in uploaded.keys() if f.lower().endswith(image_extensions)]

if len(images) < 2:
    raise ValueError("Il faut téléverser au moins deux images.")

print("Images détectées :", images)

## Choisir l'image source et l'image cible

On prend ici :
- la première image comme source ;
- la deuxième comme cible.

Vous pouvez modifier ces variables si besoin.

In [ ]:
SOURCE_IMAGE = images[0]
TARGET_IMAGE = images[1]

print("SOURCE_IMAGE =", SOURCE_IMAGE)
print("TARGET_IMAGE =", TARGET_IMAGE)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(Image.open(SOURCE_IMAGE).convert("RGB"))
plt.title("Image source")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(Image.open(TARGET_IMAGE).convert("RGB"))
plt.title("Image cible")
plt.axis("off")

plt.tight_layout()
plt.show()

## Charger les outils de détection et de swap

Le pipeline suit trois étapes :

1. détecter les visages ;
2. choisir le visage principal ;
3. appliquer le swap sur l'image cible.

In [ ]:
import cv2
from insightface.app import FaceAnalysis
from insightface.model_zoo import get_model

app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

swapper = get_model(model_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

print("Détecteur et modèle de swap chargés.")

## Détecter les visages

On détecte maintenant les visages présents dans chaque image.

### Pourquoi ?
Certaines images peuvent contenir plusieurs visages.
Pour commencer simplement, nous prendrons le plus grand visage détecté.

In [ ]:
src = cv2.imread(SOURCE_IMAGE)
tgt = cv2.imread(TARGET_IMAGE)

src_faces = app.get(src)
tgt_faces = app.get(tgt)

print("Visages détectés dans l'image source :", len(src_faces))
print("Visages détectés dans l'image cible :", len(tgt_faces))

## Choisir le visage principal

Si plusieurs visages sont présents, on prend ici le plus grand.

### Pourquoi ?
C'est souvent le visage principal de la photo.

In [ ]:
def face_area(face):
    x1, y1, x2, y2 = face.bbox
    return (x2 - x1) * (y2 - y1)

src_face = sorted(src_faces, key=face_area, reverse=True)[0]
tgt_faces_sorted = sorted(tgt_faces, key=face_area, reverse=True)

print("Visage source sélectionné.")
print("Nombre de visages cibles à traiter :", len(tgt_faces_sorted))

## Appliquer le face swap

On transfère maintenant l'identité du visage source
sur le ou les visages détectés dans l'image cible.

Pour une première démonstration, cela permet même de voir ce qui se passe
si l'image cible contient plusieurs personnes.

In [ ]:
result = tgt.copy()

for face in tgt_faces_sorted:
    result = swapper.get(result, face, src_face, paste_back=True)

out_path = "/content/faceswap_result.png"
cv2.imwrite(out_path, result)

print("Image générée :", out_path)

In [ ]:
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(result_rgb)
plt.title("Résultat du face swap")
plt.axis("off")
plt.show()

## Ce que fait réellement un face swap

Un face swap ne remplace pas toute l'image.

En général, le modèle essaie surtout de transférer :
- certains traits identitaires du visage ;
- l'apparence faciale globale.

Mais il conserve souvent :
- la pose ;
- l'éclairage ;
- l'arrière-plan ;
- une partie du contour global du visage cible.